In [ ]:
import polars as pl
from datasets import load_dataset

In [ ]:
markets = pl.read_parquet('data/processed/markets_political.parquet')

In [ ]:
for i,m in enumerate(markets.iter_rows(named=True)):
    print(i,m['condition_id'])

In [ ]:
desired_markets = markets['condition_id'].to_list()

In [ ]:
# Stream just the first batch to check schema and content
dataset = load_dataset(
    "SII-WANGZJ/Polymarket_data",
    data_files="quant.parquet",
    streaming=True,
    split="train",
    filters=[("condition_id", "in", desired_markets)]
)

In [ ]:
rows = []
for i,x in enumerate(dataset):
    rows.append(x)
    if i % 10000 == 0:
        print(f"Loaded {i} rows")

    if i == 999_000:
        break
        

In [ ]:
sample = pl.DataFrame(rows)

In [ ]:
for s in sample.group_by('condition_id').agg([
    pl.len().alias('count')
]).sort('count')[-1].iter_rows(named=True):
    print(s)

In [ ]:
import polars as pl
from datetime import timedelta

sample = pl.read_parquet('data/sample/quant_sample_political.parquet')
markets = pl.read_parquet('data/processed/markets_political.parquet')

# Find which market has the most trades in your sample
best_markets = (sample
    .group_by("condition_id")
    .len()
    .sort("len", descending=True)
    .head(10))

print("Markets with most trades in sample:")
print(best_markets)

# Use the top market
c_id = best_markets["condition_id"][0]
print(f"\nUsing condition_id: {c_id}")

# Convert timestamp to datetime
sample = sample.with_columns(
    pl.from_epoch(pl.col('timestamp'), time_unit='s').alias('datetime')
).with_columns(
    pl.col('datetime').dt.convert_time_zone('UTC').dt.cast_time_unit('ms')
)

# Filter to single market
m_sample = sample.filter(pl.col('condition_id') == c_id)

# Use latest available trade as the reference point instead of end_date
# This is only for testing on the sample — on full data you use end_date
latest_trade = m_sample['datetime'].max()
window_start = latest_trade - timedelta(days=30)

window_trades = m_sample.filter(
    (pl.col('datetime') >= window_start) &
    (pl.col('datetime') <= latest_trade)
)

print(f"Latest trade: {latest_trade}")
print(f"Window start: {window_start}")
print(f"Trades in window: {len(window_trades)}")
print(window_trades.select(['datetime', 'price', 'usd_amount', 'side']))

In [ ]:
price_series = window_trades['price']

In [ ]:
# Feature engineering

import numpy as np

# Price features
price_start = price_series.first()  
price_end = price_series.last()
price_mean = price_series.mean()
price_min = price_series.min()
price_max = price_series.max()
price_volatility = price_series.std()
price_range = price_max - price_min
price_momentum = price_end - price_start

# Volume and activity features
log_total_volume = np.log1p(window_trades['usd_amount'].sum())
log_trade_amount = np.log1p(len(window_trades))
log_avg_trade_size = np.log1p(window_trades['usd_amount'].mean())

# Sentiment feature
buy_ratio = (window_trades['side'] == 'BUY').mean()

In [ ]:
print(f"price_start:      {price_start:.4f}")
print(f"price_end:        {price_end:.4f}")
print(f"price_mean:       {price_mean:.4f}")
print(f"price_min:        {price_min:.4f}")
print(f"price_max:        {price_max:.4f}")
print(f"price_volatility: {price_volatility:.4f}")
print(f"price_range:      {price_range:.4f}")
print(f"price_momentum:   {price_momentum:.4f}")
print(f"log_total_volume:   {log_total_volume:.4f}")
print(f"log_trade_count:    {log_trade_amount:.4f}")
print(f"log_avg_trade_size: {log_avg_trade_size:.4f}")
print(f"buy_ratio:          {buy_ratio:.4f}")

In [ ]:
window_start

In [ ]:
window_trades

In [ ]:
window_trades

In [ ]:
sample_markets = sample['condition_id'].unique().to_list()

In [ ]:
duplicates = list(set(sample_markets) & set(desired_markets))

In [ ]:
len(duplicates)

In [ ]:
sample.sort('timestamp', descending=False)

In [ ]:
sample.write_parquet('data/sample/quant_sample_political.parquet')